# 데이터 구조 탐색

In [ ]:
import numpy as np
import pandas as pd
from IPython import display

train = pd.read_csv("../data/raw/train.csv").copy()

head()로 구조 확인
* 특성 개수 11개
* 범주형/숫자형/문자열 등을 확인할 수 있음
    * Pclass, Sex, Embarked, Survived
    * Cabin은 범주형인지 잘 모르겠음 -> value_counts()로 확인해봐야 알 수 있을듯
* 레이블은 Survived

In [ ]:
train.head()

info()로 데이터 결측치 유무 확인 및 데이터 타입 확인
* 행 개수는 총 891개 -> 상당히 적음을 알 수 있음
* Age, Cabin, Embarked에 결측치 존재
* 그러나 테스트셋에는 다른 열에도 결측치가 존재할 가능성을 염두에 둬야한다.

In [ ]:
train.info()

각 범주형 특성에 대해 각 범주타입 개수를 확인한다.
* Pclass -> 3,2,1있음
* Sex -> male, female
* Embarked -> S, C, Q있음

In [ ]:
from IPython.display import display
categories = ["Pclass", "Sex", "Embarked", "SibSp", "Parch"]
cat_train = train[categories].copy()
for col in categories:
    print(cat_train[col].value_counts())

In [ ]:
train.describe()

* 여기에서 describe로 딱히 알만한건 없음
* 그나마 스케일이 좀 다르다는거? -> 나머지는 히스토그램으로 확인예정
* 특히 AGe, Fare의 평균이 다른 특성에 비해 높다.
* 또한 Fare는 max값을 보면 평균에 비해 상당히 높음 -> 이상치 존재 (이상치는 아니지만 범위가 너무 넓음 근데 75%백분위수를 보면 상당수의 값은 30이하로 되어있음을 알 수 있다.

In [ ]:
import matplotlib.pyplot as plt
train.hist(bins=50, figsize=(12, 8))
plt.show()

* fare는 오른쪽 꼬리가 상당히 김 (describe에서 확인함)
* Age, Fare의 스케일 조정이 필요 (describe에서 확인함)
* 최댓값, 최솟값이 한정된 특성은 없다.


# 테스트 셋 분리
* test.csv가 존재하지만 이는 실제 제출용 데이터일 뿐 -> 실제 모델에서 사용하는 test셋으로 생각할 필요가 없음
* 어차피 학습 과정에서 k-fold를 사용하기 때문에 -> 검증 세트 자체는 필요가 없음
* 하지만 각 모델별 최종 성능을 측정할 테스트셋을 따로 분리해야 최종 모델을 선택할 수 있음
* 참고로 검증 세트는 하이퍼파라미터를 조정할때 기준점이 된다.
* **계층적 샘플링 적용(Pclass)**

train -> 	모델 파라미터 학습 (fit) (계속 사용)


validation ->   하이퍼파라미터 튜닝, 모델별 비교 (모델을 한번 학습 시킬 때 마다 검증셋을 임시 테스트셋으로 계속 사용)


(검증셋은 우연히 잘못 뽑힐 가능성으로 인해 안정적인 성능 평가를 위해 k-fold사용)


test -> 최종 모델이 결정되면 train + validation으로 전체 재학습 후 -> test셋으로 최종 성능 평가


In [ ]:
from sklearn.model_selection import train_test_split

train_set, test_set = train_test_split(
    train, test_size=0.2, stratify=train["Pclass"], random_state=42
)

print(train_set.head())
type(train_set)
train_set.to_csv("../data/processed/01/train.csv", index=False)
test_set.to_csv("../data/processed/01/test.csv", index=False)

# 나눠놓은 학습/테스트 셋 오염방지를 위해 복사하여 이후 시각화
train_set = train_set.copy()
test_set = test_set.copy()


In [ ]:
## 잘 분리 됐는지 확인
fig, axes = plt.subplots(nrows=2, figsize=(12, 8))
train_set["Pclass"].hist(ax=axes[0])
test_set["Pclass"].hist(ax=axes[1])
plt.show()

# 데이터 시각화
* 항상 y축에는 타깃이 들어가야한다. 왜? 타깃과의 연관성을 시각화하여 인사이트를 얻기 위해서..
* 타이타닉 문제는 회귀가 아닌 분류 문제이기 때문에 시각화 접근을 다르게 해야함 -> 타깃이 0/1로만 있음 따라서 y축에 타깃을 그대로 넣을 수가 없음
    * 어떻게? 1/0대신에 타깃의 평균을 넣는다.
    * 근데 타깃의 평균을 내기 위해서는 각 범주들의 그룹별로 묶은 뒤 타깃의 평균을 각각 내서 확인해야함
    * 따라서 숫자형 또한 범주형으로 변경하여 타깃의 평균을 확인한다.

정리하자면 분류 EDA는 모든 피처를 그룹으로 만들고, 그룹별 타깃 평균을 막대로 그린다.


* 또한 추가적으로 타깃을 0/1로 그룹화하여 생존자 그룹의 나이 분포/티켓 가격 분포, 비생존자 그룹의 .. 을 확인해야한다.





1. 모든 특성을 범주형으로 변경한다.
2. 그룹별 생존율을 확인해야함
    1. 가장 생존율이 차이가 나는 특성을 우선순위로 시각화한다.
    2. 또한 타깃에 강하게 연관된 특성에 대해 각 조합을 확인한다.
        1. 이때 가장 강하게 연관된 특성 하나를 잡고 나머지 특성에 대해 조합해보는게 빠르고 효율적
3. 범주형 -> 범주별 생존 여부의 수치를 막대그래프로 확인
4. 숫자형 -> 타깃값을 그룹화하여 생존자/비생존자의 숫자형 특성의 분포를 확인하라
5. 타깃과 강하게 연관된 특성끼리의 조합을 시각화하여 더 깊게 탐색

### 1. 숫자형에 대해 범주형 특성 새로 생성

In [ ]:
train_set["cat_age"] = pd.cut(train_set["Age"], bins=[0,10, 15,20,30,40,60,80,100])
train_set["cat_fare"] = pd.qcut(train_set["Fare"], q=4)

### 2. 그룹별 생존율 확인

In [ ]:
categories = ["Pclass", "Sex", "Embarked", "SibSp", "Parch", "cat_age", "cat_fare"]
for col in categories:
    print(train_set.groupby(col)["Survived"].mean().sort_values(ascending=False))

* Embarked 특성은 크게 타깃에 연관이 없는 듯 하다
* SibSp, Parch는 연관이 있어보이나 생존율이 낮은 범주들은 모두 개체 개수 자체가 적기 때문에 학습셋에 과대적합될 가능성이 존재한다. -> 나중에 특성을 가공하여 생존율과의 연관을 확인 후 사용
* 의외로 age, fare를 범주화한 특성이 생존율에 강한 연관을 가지는 듯 하다
    * age특성에서는 의외로 15\~20살, 20\~30대의 나이대가 생존율이 낮은 경향을 보이기 때문
        * 아마 구출 과정에서 젊은 나이대가 구출을 돕는 경향이 높아서? 인듯
        * 나중에 성별과 엮어서 확인해보면 더 좋은 연관을 얻을 수 있을 것 같다.
    * fare 특성은 직관 그대로 티켓 가격이 높을수록 생존율이 높은 관계를 확인할 수 있음

### 3. 범주형 특성별 생존율 시각화
-> 위의 그룹별 생존율 표로도 확인이 가능하지만 그냥 나중에 다시 봐도 헷갈리지 않게 하려고..

In [ ]:
train_set.groupby("Pclass")["Survived"].mean().plot(kind="bar")
plt.show()

In [ ]:
train_set.groupby("Sex")["Survived"].mean().plot(kind="bar")

In [ ]:
train_set.groupby("cat_age")["Survived"].mean().plot(kind="bar")
plt.show()

In [ ]:
train_set.groupby("cat_fare")["Survived"].mean().plot(kind="bar")
plt.show()

### 4. 가장 강하게 연관된 특성에 대해 모든 조합의 생존율 확인하기
* 컬럼 Sex사용
* dataframe클래스 함수중 pivot_table함수 사용
    * index -> 세로축에 둘 컬럼을 지정한다.
    * columns -> 가로축에 둘 컬럼을 지정한다.
        *  -> index + columns 조합으로 그룹화가 된다.
        * 다시 말해 index컬럼별로 먼저 그룹화가 되고 각 그룹내에서도 columns별로 그룹화가 되어 지정한 value의 값이 나옴
        * value -> 그룹화된 값들의 조회할 값 지정 (기본 값 계산방법은 평균을 구하기 때문에 Survived행을 지정하면 각 그룹의 Survived컬럼의 평균값이 나온다고 생각하면 된다.)


* 또한 각 조합의 그룹별로 개체의 개수도 확인해야한다. 왜? -> 여성 + 80세의 생존율이 0.98로 나와도 해당 그룹의 개체 개수가 3명이라면??
    * 30명 미만 → 참고만, 결론 내리지 말 것
    * 10명 미만 → 무시
    * 100명 이상 → 안정적

1. Sex + Pclass 조합
    * 1등석, 2등석 + 여성은 가장 생존율이 높음
    * 3등석부터 여성은 급격하게 생존율이 낮아짐
    * 전체적으로 남성보다 여성이 생존율이 상당히 높은 모습


특성 조합에 따라 생존율이 높아지는 특정 조합을 발견할 수 없음. 단순히 성별과 좌석 등급의 추이를 따라가는 모습이다.

-> 등급이 낮아질 수록 생존율이 떨어지는 모습

-> 또한 성별의 차이에 따라 전체적인 생존율 차이가 크게 남

In [ ]:
table = train_set.pivot_table(index="Pclass", columns="Sex", values="Survived" ,aggfunc=["mean", "count"])
display(table)

table["mean"].plot(kind="line")
plt.show()

###계속 헷갈리는데 각 컬럼별로 그래프 하나가 생기는 것
###그리고 각 값들이 y축이 되고 행 하나가 x축 하나라고 생각하면 된다.

2. Sex + cat_age조합
* 전체적인 성별에 따른 생존율 차이는 동일하나 나이가 높아질 수록 성별에 따라 생존율 추이가 다르다 -> 주목할만한 지점
* 그러나 각 그룹의 사람 수가 적기 때문에 단순 성인 vs 아이로 나눠도 될 것 같다.

In [ ]:
table = train_set.pivot_table(index="cat_age", columns="Sex", values="Survived", aggfunc=["mean", "count"])
display(table)

table["mean"].plot(kind="line")
plt.show()

2번 조합에 대한 추가적인 조사

-> 나이의 범위 자체를 더 넓게 잡았더니 생존율의 큰 차이가 보이지 않는다. 오히려 성별에 따른 생존율 차이만 더 돋보인다.. 뭐지

In [ ]:
train_set["isAdult"] = pd.cut(train_set["Age"], bins=[0,20, 40, 100])
table = train_set.pivot_table(index="isAdult", columns="Sex", values="Survived" ,aggfunc=["mean", "count"])
display(table)
table["mean"].plot(kind="line")
plt.show()



3. Sex + cat_fare 조합
* 신기하게 여성은 특정 티켓 가격까지는 생존율의 큰 차이를 보이지 않지만 특정 구간 이상부터 생존율이 높음
    * 티켓 가격 구간을 더 잘게 잘라서 추이를 확인해봐야겠다
        * 아 근데 이러면 그룹 별 개체 수가 적어져서 생존율 값의 의미가 없어짐
* 남성 그룹은 단순히 티켓가격이 높아질 수록 생존율이 올라가는 모습
* 또한 알 수 있는 점은 여성은 티켓가격이 높아질 수록 인원수가 많아지고, 남성은 티켓 가격이 낮아질 수록 많아지는 경향을 확인할 수 있음

In [ ]:
table = train_set.pivot_table(index="cat_fare", columns="Sex", values="Survived", aggfunc=["mean", "count"])
display(table)

table["mean"].plot(kind="line")
plt.show()

### 5. 생존자 그룹에 대해 숫자형 특성 시각화하기
-> 생존자 그룹에 대해 나이의 분포를 확인한다. (비생존자도 동일하게)


-> 생존자 그룹에 대해 가격의 분포를 확인한다.

In [ ]:
train_set.groupby("Survived")["Age"].hist(bins=30, alpha=0.5, legend=True)
plt.xlabel("Age")
plt.show()

* 비생존자/생존자 그룹의 나이 분포대로 비슷한 경향을 보이나 어린이의 생존율이 상당히 높은 점은 주목할만함
*

In [ ]:
train_set["sqrt(fare)"] = train_set["Fare"].apply(np.sqrt)
train_set.groupby("Survived")["sqrt(fare)"].hist(bins=30, alpha=0.5, legend=True)
plt.xlabel("Fare")
plt.show()

* Fare의 스케일처리를 위해 모든 값에 sqrt함수를 적용후 생존자/비생존자 그룹으로 나누어 분포 확인
* 특정 가격대의 비생존자가 급격히 높은걸 확인할 수 있음
* 가격대가 높을 수록 생존자가 비생존자보다 많이 분포함을 알 수 있다. (오른쪽 꼬리가 길다)
* Pclass의 분포와 동일하므로 나중에 특성은 하나만 사용해도 될 듯? 이상치는 없고 Pclass의 추이를 따라가니까..

# 안해본 특성들 추가로 생각날 때 마다 분석하기

### 번외로 "SibSp", "Parch" 특성 조사하기

* SibSp -> 형제 자매와 배우자의 수 (본인 기준)
* Parch -> 부모 + 자식의 수 (본인 기준)

### 사용할 만한 가설
* 가족의 수로 합쳐서 어떤 추이를 확인할 수 있지 않을까?
* cat_age와 Parch의 조합으로 어린이 + 부모가 있으면 어린이는 생존율이 높아지는 경향을 확인할 수 있지 않을까?

In [ ]:
fig, ax = plt.subplots(nrows=3, sharex=True, figsize=(12, 13))


# 생존율
train_set.groupby("Parch")["Survived"].mean().plot(kind="bar", ax=ax[0])

# 전체 숫자별 승객 수
train_set["Parch"].value_counts().sort_index().plot(kind="bar", ax=ax[1])

# 생존자 수
train_set.loc[train_set["Survived"]==1, "Parch"].value_counts().sort_index().plot(kind="bar", ax=ax[2])
plt.show()

* 나이와 조합해야 더 자세하게 확인할 수 있을 것 같음
* 사람 수까지 확인했는데 잘 모르겠음 ..

In [ ]:
fig, ax = plt.subplots(nrows=3, sharex=True, figsize=(12, 13))


# 생존율
train_set.groupby("SibSp")["Survived"].mean().plot(kind="bar", ax=ax[0])

# 그룹별 승객 수
train_set["SibSp"].value_counts().sort_index().plot(kind="bar", ax=ax[1])

# 생존자 수
# train_set.loc[train_set["Survived"]==1, "Parch"].value_counts().sort_index().plot(kind="bar", ax=ax[2])
plt.show()

In [ ]:
table = train_set.pivot_table(index='Parch', columns='isAdult', values='Survived', aggfunc=["mean", "count"])
table["mean"].iloc[:4].plot(kind="line")
plt.show()
print(table.iloc[:4])

가설을 어린아이 + 부모가 많을 수록 생존율이 올라갈 것이다? 라고 생각을 했는데


-> Parch그룹별로 모두 나이대에 따른 생존율 차이는 크게 나지 않음

-> 어린아이도 부모가 많아질 수록 생존율이 많이 올라가지 않음 단순 Parch가 0이냐 아니냐에 따라 차이가 날 뿐

-> 어린아이만 그런게 아니라 모든 나이대의 그룹이 Parch가 많아질 수록 생존율이 높아지는 것도 아니고 0이냐 아니냐의 차이로만 두드러짐

-> Parch가 2일때 20대 그룹이 0.7로 튀는걸 확인할 수 있지만 그룹 크기가 19밖에 안되기 때문에 의미있는 값은 아님



**따라서 나이대 + Parch의 조합으로는 큰 차이를 찾을 수 없고 단순 Parch의 0/1 범주형으로 나눠도 될 것 같다.**

SipSp + Parch를 합친 가족 수

In [ ]:
train_set["family"] = train_set["SibSp"] + train_set["Parch"]
train_set["family"] = pd.cut(train_set["family"], bins=[0,1,2,3,10])
train_set.groupby("family")["Survived"].mean().plot(kind="line")
plt.show()
train_set[["family"]].value_counts().sort_index()


family는 안써도 될듯 너무 난해하게 나온다..

### 이름 특성 분석하기

In [ ]:
train_set["Name"].head()


In [ ]:
train_set["Name"].info()


In [ ]:
train_set["cat_name"] = train_set["Name"].str.extract(r",\s*([^\.]+)\.")
train_set["cat_name"].value_counts()

* Mr, Miss, Mrs, Master 를 제외하고는 모두 Rare로 처리해도 될듯

In [ ]:
common = ["Mr", "Miss", "Mrs", "Master"]
train_set["cat_name"] = train_set["cat_name"].where(train_set["cat_name"].isin(common), "Rare")

In [ ]:
train_set.groupby("cat_name")["Survived"].mean().plot(kind="bar")

* 도메인 지식인데 Master는 어린 아이를 부르는 경칭이라고 한다
* 생존율에 꽤나 연관이 있어보이므로 나중에 사용 특성으로 고려해도 좋을 것 같다.

### Cabin 특성 조사해보기
* 타이타닉 글을 조금 찾아봤는데 Cabin이 Nan인 승객은 애초에 3등석이라 객실도 없던거 -> 따라서 Nan자체를 정보로 사용해도 좋을 것 같음
* 또한 객실이 좋은 곳일 수록 구명정과 가깝다는 정보를 얻음 -> 생존율에 큰 영향을 미칠 것 같다

In [ ]:
cabin = train_set[["Cabin"]]
cabin.describe()

In [ ]:

print(cabin.isna().sum())
cabin.value_counts()

범주형이라고 하기엔 값이 너무 많으므로 앞의 영어를 기준으로 범주를 나누자..

In [ ]:
cabin["Cabin"] = cabin["Cabin"].fillna("N")
cabin["cat_cabin"] = cabin["Cabin"].str[0]
cabin["cat_cabin"].value_counts().sort_index()

단순 객실 등급과 생존율 차이 확인

-> 크게 차이는 안나보임 오히려 Nan객실과 객실이 있는 사람간의 차이만 두드러져보임

In [ ]:
cabin["Survived"] = train_set["Survived"]
cabin.groupby("cat_cabin")["Survived"].mean().plot(kind="bar")

In [ ]:
cabin["has_cabin"] = np.where(cabin["Cabin"] == 'N', 0, 1)
print(cabin["has_cabin"].value_counts().sort_index())
cabin.groupby("has_cabin")["Survived"].mean().plot(kind="bar")


-> 좋은 특성..